## Read the first few lines of the Amazon Snap File to understand the structure

In [24]:
import pandas as pd
import gzip
from urllib.request import urlopen
import io

url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"

with urlopen(url) as response:
#with gzip.open(file_path, 'rt', encoding='latin-1') as f:
    with gzip.open(response, mode='rt', encoding='latin-1') as f:
        for i in range(22):
            print(f.readline().strip())

# Full information about Amazon Share the Love products
Total items: 548552

Id:   0
ASIN: 0771044445
discontinued product

Id:   1
ASIN: 0827229534
title: Patterns of Preaching: A Sermon Sampler
group: Book
salesrank: 396585
similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X
categories: 2
|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]
|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]
reviews: total: 2  downloaded: 2  avg rating: 5
2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9
2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5

Id:   2
ASIN: 0738700797


## Download the data and store in a dataframe and save to a Parquet file with compression

In [ ]:
import pandas as pd
import gzip
from urllib.request import urlopen
import io

url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"
#file_path = r"../data/amazon-meta.txt.gz"


products = []
current_item = {}

with urlopen(url) as response:
#with gzip.open(file_path, 'rt', encoding='latin-1') as f:
    with gzip.open(response, mode='rt', encoding='latin-1') as f:
        for line in f:
            line = line.strip()

            if line.startswith("Id:"):
                if current_item:
                    products.append(current_item)
                current_item = {'Id': line.split(":")[1].strip()}
            elif line.startswith("discontinued"):
                current_item['Active'] = 'N'
            elif line.startswith("ASIN"):
                current_item['ASIN'] = line.split(":")[1].strip()
            elif line.startswith("title"):
                current_item['Title'] = line.split(":", 1)[1].strip()
            elif line.startswith("group"):
                current_item['Group'] = line.split(":")[1].strip()
            elif line.startswith("salesrank"):
                current_item['Salesrank'] = line.split(":")[1].strip()
            elif line.startswith('similar:'):
                parts = line.split()  # Splits by any whitespace
                current_item["similarCount"] = parts[1]
                current_item['SimilarASINs'] = parts[2:]
            elif line.startswith('|'):
                if 'Categories' not in current_item:
                    current_item['Categories'] = []
                    current_item['CleanCategories'] = []
                current_item['Categories'].append(line)
                # 4 levels
                line_clean = [cat.split('[')[0] for cat in line.strip('|').split('|')][:4]
                current_item['CleanCategories'].append(" > ".join(line_clean))
            elif line.startswith('reviews:'):
                parts = line.split()
                try:
                    # 'total:' is usually at index 1, so the number is at index 2
                    current_item['reviews_count'] = int(parts[2])
                    # 'rating:' is usually at index 7, so the number is at index 8
                    current_item['avg_rating'] = float(parts[8])
                except (ValueError, IndexError):
                    current_item['reviews_count'] = 0
                    current_item['avg_rating'] = 0.0

df = pd.DataFrame(products)
df.to_parquet('../data/amazon_data.parquet', compression='gzip')
print(df.head())

  Id        ASIN Active                                              Title  \
0  0  0771044445      N                                                NaN   
1  1  0827229534    NaN            Patterns of Preaching: A Sermon Sampler   
2  2  0738700797    NaN                         Candlemas: Feast of Flames   
3  3  0486287785    NaN   World War II Allied Fighter Planes Trading Cards   
4  4  0842328327    NaN  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0   NaN       NaN          NaN   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                                NaN   
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
3                                                 []   
4  [0842328130, 0830818138, 0842330313, 084232

In [30]:
import pandas as pd

file_path = r"../data/amazon_data.parquet"
df = pd.read_parquet(file_path)
print(df.head())
print("Data loaded successfully!")

  Id        ASIN Active                                              Title  \
0  0  0771044445      N                                               None   
1  1  0827229534   None            Patterns of Preaching: A Sermon Sampler   
2  2  0738700797   None                         Candlemas: Feast of Flames   
3  3  0486287785   None   World War II Allied Fighter Planes Trading Cards   
4  4  0842328327   None  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0  None      None         None   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                               None   
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
3                                                 []   
4  [0842328130, 0830818138, 0842330313, 084232